In [56]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [57]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [58]:
#import dataset
youtube_shorts_tiktok_trends_2025 = pd.read_csv('/content/drive/MyDrive/raw_data/youtube_shorts_tiktok_trends_2025.csv')

In [59]:
#read dataset
print(youtube_shorts_tiktok_trends_2025.head(5))
print(youtube_shorts_tiktok_trends_2025.info())

  platform country   region language category     hashtag  \
0   TikTok      Jp     Asia       ja   Gaming  #Lifestyle   
1   TikTok      Se   Europe       sv     Food     #Sports   
2   TikTok      Za   Africa       en      Art    #Workout   
3   TikTok      Kr     Asia       ko     News    #Esports   
4   TikTok      Au  Oceania       en   Beauty     #Comedy   

                  title_keywords    author_handle sound_type     music_track  \
0        Night Routine — College       NextVision   trending       8bit loop   
1      Morning Routine — College  DailyVlogsDiego   trending     Street vibe   
2        Night Routine — College        BeyondHub   licensed     Gallery pad   
3     Best Settings for Fortnite          NextHub   original   Neutral piano   
4  When your friend is Beginners    LucasOfficial   licensed  Soft glam loop   

   ...  traffic_source  is_weekend                            row_id  \
0  ...        External           1  2e681528d17a1fe1986857942536ec27   
1  ...  

In [60]:
#checking dataset
youtube_shorts_tiktok_trends_2025.isnull().sum()

,0
platform,0
country,0
region,0
language,0
category,0
hashtag,0
title_keywords,0
author_handle,0
sound_type,0
music_track,0


In [61]:
#EDA: What Factors Drive Viral Short-Form Content?

#separate tiktok vs youtube
youtube_shorts_trend_2025 = youtube_shorts_tiktok_trends_2025[youtube_shorts_tiktok_trends_2025['platform'] == 'YouTube']
tiktok_shorts_trend_2025 = youtube_shorts_tiktok_trends_2025[youtube_shorts_tiktok_trends_2025['platform'] == 'TikTok']

In [62]:
#calculate p90 each platform
p90_youtube = youtube_shorts_trend_2025['engagement_velocity'].quantile(0.9)
p90_tiktok = tiktok_shorts_trend_2025['engagement_velocity'].quantile(0.9)
print(p90_youtube)
print(p90_tiktok)

31197.030000000013
33352.52500000001


In [63]:
#create viral_trend each platform
def youtube_viral(row):
  if row['engagement_velocity'] >= p90_youtube:
    return 'viral'
  else:
    return 'no_viral'

def tiktok_viral(row):
  if row['engagement_velocity'] >= p90_tiktok:
    return 'viral'
  else:
    return 'no_viral'

youtube_shorts_trend_2025['viral_trend'] = youtube_shorts_trend_2025.apply(youtube_viral, axis=1)
tiktok_shorts_trend_2025['viral_trend'] = tiktok_shorts_trend_2025.apply(tiktok_viral, axis=1)

/tmp/ipykernel_2155/3834892892.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  youtube_shorts_trend_2025['viral_trend'] = youtube_shorts_trend_2025.apply(youtube_viral, axis=1)
/tmp/ipykernel_2155/3834892892.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tiktok_shorts_trend_2025['viral_trend'] = tiktok_shorts_trend_2025.apply(tiktok_viral, axis=1)


In [64]:
#EDA: What Factors Drive Viral Short-Form YouTube Content?
#numeric variables

numeric_factors_youtube = youtube_shorts_trend_2025.groupby('viral_trend').agg(
    {
        'completion_rate' : 'median',
        'share_rate' : 'median',
        'engagement_velocity' : 'median',
        'upload_hour' : 'median'
    }

).reset_index()
numeric_factors_youtube.columns = ['viral_trend', 'completion_rate', 'share_rate', 'engagement_velocity', 'upload_hour']
print(numeric_factors_youtube)

#categorical_variables
genre_youtube = pd.crosstab(youtube_shorts_trend_2025['genre'], youtube_shorts_trend_2025['viral_trend'])
genre_youtube['viral_rate'] = genre_youtube['viral'] / (genre_youtube['viral'] + genre_youtube['no_viral'])*100
genre_youtube['viral_rate'] = genre_youtube['viral_rate'].round(2)
print(genre_youtube.sort_values(by='viral_rate', ascending=False).head(5))

creator_youtube = pd.crosstab(youtube_shorts_trend_2025['creator_tier'], youtube_shorts_trend_2025['viral_trend'])
creator_youtube['viral_rate'] = creator_youtube['viral'] / (creator_youtube['viral'] + creator_youtube['no_viral'])*100
creator_youtube['viral_rate'] = creator_youtube['viral_rate'].round(2)
print(creator_youtube.sort_values(by='viral_rate', ascending=False))

sound_youtube = pd.crosstab(youtube_shorts_trend_2025['sound_type'], youtube_shorts_trend_2025['viral_trend'])
sound_youtube['viral_rate'] = sound_youtube['viral'] / (sound_youtube['viral'] + sound_youtube['no_viral'])*100
sound_youtube['viral_rate'] = sound_youtube['viral_rate'].round(2)
print(sound_youtube)

  viral_trend  completion_rate  share_rate  engagement_velocity  upload_hour
0    no_viral            0.574    0.003536             5435.670         17.0
1       viral            0.576    0.003550            48610.475         17.0
viral_trend  no_viral  viral  viral_rate
genre                                   
DIY                95     12       11.21
Music            1334    162       10.83
Beauty           1355    159       10.50
Lifestyle        1263    148       10.49
Travel           1345    154       10.27
viral_trend   no_viral  viral  viral_rate
creator_tier                             
Mid              17294   1924       10.01
Micro               17      0        0.00
viral_trend  no_viral  viral  viral_rate
sound_type                              
licensed         5783    636        9.91
original         5861    649        9.97
trending         5667    639       10.13


In [65]:
#EDA: What Factors Drive Viral Short-Form TikTok Content?
#numeric variables

numeric_factors_tiktok = tiktok_shorts_trend_2025.groupby('viral_trend').agg(
    {
        'completion_rate' : 'median',
        'share_rate' : 'median',
        'engagement_velocity' : 'median',
        'upload_hour' : 'median'
    }
).reset_index()
numeric_factors_tiktok.columns = ['viral_trend', 'completion_rate', 'share_rate', 'engagement_velocity', 'upload_hour']
print(numeric_factors_tiktok)

#categorical_variables
genre_tiktok = pd.crosstab(tiktok_shorts_trend_2025['genre'], tiktok_shorts_trend_2025['viral_trend'])
genre_tiktok['viral_rate'] = genre_tiktok['viral'] / (genre_tiktok['viral'] + genre_tiktok['no_viral'])*100
genre_tiktok['viral_rate'] = genre_tiktok['viral_rate'].round(2)
print(genre_tiktok.sort_values(by='viral_rate', ascending=False).head(5))

creator_tiktok = pd.crosstab(tiktok_shorts_trend_2025['creator_tier'], tiktok_shorts_trend_2025['viral_trend'])
creator_tiktok['viral_rate'] = creator_tiktok['viral'] / (creator_tiktok['viral'] + creator_tiktok['no_viral'])*100
creator_tiktok['viral_rate'] = creator_tiktok['viral_rate'].round(2)
print(creator_tiktok.sort_values(by='viral_rate', ascending=False))

sound_tiktok = pd.crosstab(tiktok_shorts_trend_2025['sound_type'], tiktok_shorts_trend_2025['viral_trend'])
sound_tiktok['viral_rate'] = sound_tiktok['viral'] / (sound_tiktok['viral'] + sound_tiktok['no_viral'])*100
sound_tiktok['viral_rate'] = sound_tiktok['viral_rate'].round(2)
print(sound_tiktok)

  viral_trend  completion_rate  share_rate  engagement_velocity  upload_hour
0    no_viral            0.677    0.006412              5574.67         17.0
1       viral            0.679    0.006409             51415.00         17.0
viral_trend  no_viral  viral  viral_rate
genre                                   
DIY               125     17       11.97
Music            1914    241       11.18
Lifestyle        1922    233       10.81
Education        2016    244       10.80
Fashion          1961    235       10.70
viral_trend   no_viral  viral  viral_rate
creator_tier                             
Mid              25943   2885       10.01
Micro               16      0        0.00
viral_trend  no_viral  viral  viral_rate
sound_type                              
licensed         8617    964       10.06
original         8661    931        9.71
trending         8681    990       10.24


In [66]:
#EDA: TikTok VS YouTube Perfomances

tiktok_vs_youtube = youtube_shorts_tiktok_trends_2025.groupby('platform').agg(
    {
        'engagement_rate' : 'median',
        'completion_rate' : 'median',
        'share_rate' : 'median',
        'avg_watch_time_sec' : 'median'
    }
).reset_index()

tiktok_vs_youtube.columns = ['platform', 'engagement_rate', 'completion_rate', 'share_rate', 'avg_watch_time_sec']
print(tiktok_vs_youtube)

  platform  engagement_rate  completion_rate  share_rate  avg_watch_time_sec
0   TikTok         0.088813            0.677    0.006412                18.4
1  YouTube         0.049080            0.574    0.003538                21.2


In [67]:
#EDA: YouTube Best Time To Upload
youtube_best_time_upload = youtube_shorts_trend_2025.groupby(['publish_dayofweek', 'publish_period', 'upload_hour'])['engagement_velocity'].median().reset_index()
youtube_best_time_upload.columns = ['publish_dayofweek', 'publish_period', 'upload_hour', 'engagement_velocity']
print(youtube_best_time_upload.sort_values(by='engagement_velocity', ascending=False).head(5))

   publish_dayofweek publish_period  upload_hour  engagement_velocity
35            Monday        Morning            5            13417.415
92            Sunday          Night            2            12659.430
90            Sunday          Night            0            12634.905
69          Saturday          Night            3            11210.035
42            Monday          Night            0            10852.500


In [68]:
#EDA: TikTok Best Time To Upload
tiktok_best_time_upload = tiktok_shorts_trend_2025.groupby(['publish_dayofweek', 'publish_period', 'upload_hour'])['engagement_velocity'].median().reset_index()
tiktok_best_time_upload.columns = ['publish_dayofweek', 'publish_period', 'upload_hour', 'engagement_velocity']
print(tiktok_best_time_upload.sort_values(by='engagement_velocity', ascending=False).head(5))

    publish_dayofweek publish_period  upload_hour  engagement_velocity
142           Tuesday          Night            4             11884.00
90             Sunday          Night            0             10163.00
93             Sunday          Night            3              9288.68
91             Sunday          Night            1              9008.60
162         Wednesday          Night            0              8794.94


In [69]:
#YouTube Viral Category
youtube_viral_category = pd.crosstab(youtube_shorts_trend_2025['category'], youtube_shorts_trend_2025['viral_trend'])
youtube_viral_category['viral_rate'] = youtube_viral_category['viral'] / (youtube_viral_category['viral'] + youtube_viral_category['no_viral'])*100
youtube_viral_category['viral_rate'] = youtube_viral_category['viral_rate'].round(2)
print(youtube_viral_category.sort_values(by='viral_rate', ascending=False).head(5))

viral_trend  no_viral  viral  viral_rate
category                                
Finance           881    114       11.46
Comedy            875    111       11.26
Science           911    114       11.12
Art               851    106       11.08
Fitness           929    113       10.84


In [70]:
#TikTok Viral Categpry
tiktok_viral_category = pd.crosstab(tiktok_shorts_trend_2025['category'], tiktok_shorts_trend_2025['viral_trend'])
tiktok_viral_category['viral_rate'] = tiktok_viral_category['viral'] / (tiktok_viral_category['viral'] + tiktok_viral_category['no_viral'])*100
tiktok_viral_category['viral_rate'] = tiktok_viral_category['viral_rate'].round(2)
print(tiktok_viral_category.sort_values(by='viral_rate', ascending=False).head(5))

viral_trend  no_viral  viral  viral_rate
category                                
DIY              1311    173       11.66
Gaming           1313    160       10.86
Music            1340    162       10.79
Automotive       1358    164       10.78
Education        1329    160       10.75


In [71]:
#engagement velocity analysis
#YouTube
youtube_velocity = youtube_shorts_trend_2025.groupby(['viral_trend', 'trend_label', 'trend_type'])[['engagement_velocity', 'trend_duration_days']].median().reset_index()
youtube_velocity.columns = ['viral_trend', 'trend_label', 'trend_type', 'engagement_velocity', 'trend_duration_days']
print(youtube_velocity.sort_values(by='engagement_velocity', ascending=False))

   viral_trend trend_label trend_type  engagement_velocity  \
12       viral   declining  Evergreen            82983.590   
18       viral    seasonal  Evergreen            66039.300   
15       viral      rising  Evergreen            52372.090   
20       viral    seasonal      Short            50292.330   
23       viral      stable      Short            50037.670   
17       viral      rising      Short            49973.000   
14       viral   declining      Short            49544.750   
16       viral      rising     Medium            41148.250   
19       viral    seasonal     Medium            40769.700   
13       viral   declining     Medium            38880.900   
22       viral      stable     Medium            37544.410   
21       viral      stable  Evergreen            34863.805   
8     no_viral    seasonal      Short            10098.150   
5     no_viral      rising      Short             9837.000   
11    no_viral      stable      Short             9652.345   
2     no

In [72]:
#engagement velocity analysis
#TikTok
tiktok_velocity = tiktok_shorts_trend_2025.groupby(['viral_trend', 'trend_label', 'trend_type'])[['engagement_velocity', 'trend_duration_days']].median().reset_index()
tiktok_velocity.columns = ['viral_trend', 'trend_label', 'trend_type', 'engagement_velocity', 'trend_duration_days']
print(tiktok_velocity.sort_values(by='engagement_velocity', ascending=False))

   viral_trend trend_label trend_type  engagement_velocity  \
18       viral    seasonal  Evergreen            62602.350   
23       viral      stable      Short            52852.500   
17       viral      rising      Short            52296.330   
14       viral   declining      Short            51967.500   
20       viral    seasonal      Short            50763.830   
13       viral   declining     Medium            47720.490   
22       viral      stable     Medium            47364.805   
19       viral    seasonal     Medium            44413.000   
16       viral      rising     Medium            42438.400   
15       viral      rising  Evergreen            37843.100   
12       viral   declining  Evergreen            37695.200   
21       viral      stable  Evergreen            34353.895   
8     no_viral    seasonal      Short            10008.870   
11    no_viral      stable      Short             9972.215   
2     no_viral   declining      Short             9934.270   
5     no